# Prerequirements & Setup
Start by making sure you have the required files in the proper structure before trying this, because it requires you to mount the drive in order to access the python scripts and datasets. The files should be accessible via our repo.

## Installing Dependencies
First, we need to install the required Python packages that are not pre-installed in Google Colab. This includes `transformers` for the language models, `datasets` for loading data, and `openai` (if your scripts interact with OpenAI's API).

In [1]:
# Install necessary dependencies not pre-installed in Colab
!pip install transformers datasets openai

In [4]:
# Consolidated imports from run.py, custom_datasets.py, and calculate_metrics.py
import argparse
import datetime
import functools
import json
import os
import random
import re
import time
from multiprocessing.pool import ThreadPool

import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import tqdm
import transformers

from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve
)

In [5]:
# Check if PyTorch is using the GPU version and if a GPU is available
print("PyTorch version:", torch.__version__)
print("Is CUDA (GPU) available?", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu128
Is CUDA (GPU) available? True
GPU Device Name: Tesla T4


## Mounting Google Drive
Since the project files (like `run.py`) and datasets are stored in your Google Drive, we need to mount it to the Colab environment. When you run this, Colab will prompt you to grant access to your Drive.

**Note:** In the code cell after this one, you MUST update the `project_path` variable to point to the exact folder where you uploaded the DetectGPT files.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os

# Define the path to your project folder. Where ever you set your project folder to be, set it here
# Example:
# project_path = "/content/drive/MyDrive/{YOUR FOLDER LOCATION}"
project_path = "/content/drive/MyDrive/CSCI 544: Applied Natural Language Processing/Project/DetectGPT"


# Change the current working directory to your project folder
os.chdir(project_path)
print("Current Working Directory:", os.getcwd())

# Set up a cache directory in your drive for HuggingFace models
cache_dir = os.path.join(project_path, "hf_cache")
os.makedirs(cache_dir, exist_ok=True)
os.environ['HF_HOME'] = cache_dir

print("HuggingFace cache set to:", os.environ['HF_HOME'])

Current Working Directory: /content/drive/MyDrive/CSCI 544: Applied Natural Language Processing/Project/DetectGPT
HuggingFace cache set to: /content/drive/MyDrive/CSCI 544: Applied Natural Language Processing/Project/DetectGPT/hf_cache


# Running DetectGPT

## Run Flags Reference

| Flag | Required | Example | Meaning |
|---|---|---|---|
| --output_name | Recommended | smoke_hc3 | Name of the run output folder so results are easy to identify. |
| --dataset | Yes | hc3_all | Dataset key to load and evaluate. |
| --base_model_name | Yes | gpt2-medium | Base causal language model used for generation and scoring. |
| --mask_filling_model_name | Yes (for perturbation runs) | t5-base | Mask infilling model used to create perturbed text. |
| --n_perturbation_list | Yes (for perturbation runs) | 3 or 1,3,10 | Number of perturbations per sample. You can pass a comma-separated list. |
| --n_samples | Yes | 50 | Number of examples to evaluate. Smaller is faster. |
| --pct_words_masked | Recommended | 0.3 | Approximate fraction of words to mask before refill perturbation. |
| --skip_baselines | Optional switch | include flag only | If present, skips baseline metrics and runs perturbation experiments only. |
| --cache_dir | Recommended | hf_cache (if running in colab) or C:/hf_home (if running on local) | Directory for model/dataset cache to avoid repeated downloads. |

## Other Useful optional flags:

| Flag | Needed | Example value | What it means |
|---|---|---|---|
| --batch_size | Optional | 10 | Batch size for scoring/supervised loops. Lower if memory is tight. |
| --chunk_size | Optional | 10 | Chunk size for perturbation generation. Lower if VRAM/RAM is limited. |
| --span_length | Optional | 2 | Number of contiguous tokens per masked span. |
| --n_perturbation_rounds | Optional | 1 | How many times perturbation is recursively applied. |

## Small Smoke Test Command
```shell
python run.py --output_name smoke_hc3 --dataset hc3_all --base_model_name gpt2 --mask_filling_model_name t5-small --n_perturbation_list 3 --n_samples 50 --pct_words_masked 0.3 --skip_baselines --cache_dir C:/hf_home --batch_size 10 --chunk_size 10
```
## Reusable Template
```shell
python run.py --output_name YOUR_RUN_NAME --dataset YOUR_DATASET --base_model_name gpt2-medium --mask_filling_model_name t5-base --n_perturbation_list 3 --n_samples 500 --pct_words_masked 0.3 --skip_baselines --cache_dir hf_cache
```





## Smoke Test 1: Testing using the original dataset (Writing Prompt)
This is a small "smoke test" designed to run quickly and verify that your environment is set up correctly without errors. It processes only 50 samples from the `writing` dataset.

Flags used in this run:

* `--output_name tiny_50`
* `--dataset writing`
* `--base_model_name gpt2`
* `--mask_filling_model_name t5-small`
* `--n_perturbation_list 5`
* `--n_samples 50`
* `--pct_words_masked 0.3`
* `--span_length 2`
* `--batch_size 4`
* `--chunk_size 4`
* `--skip_baselines`
* `--cache_dir hf_cache`

In [ ]:
!python run.py --output_name tiny_50 --dataset writing --base_model_name gpt2 --mask_filling_model_name t5-small --n_perturbation_list 5 --n_samples 50 --pct_words_masked 0.3 --span_length 2 --batch_size 4 --chunk_size 4 --skip_baselines --cache_dir hf_cache

Using device: cuda
Saving results to absolute path: /content/drive/MyDrive/CSCI 544: Applied Natural Language Processing/Project/DetectGPT/tmp_results/tiny_50/gpt2-t5-small-temp/2026-04-11-22-01-02-885558-fp32-0.3-1-writing-50
Using cache dir hf_cache
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 128kB/s]
vocab.json: 1.04MB [00:00, 20.1MB/s]
merges.txt: 456kB [00:00, 4.93MB/s]
tokenizer.json: 1.36MB [00:00, 21.0MB/s]
Loading BASE model gpt2...
config.json: 100% 665/665 [00:00<00:00, 3.35MB/s]
model.safetensors: 100% 548M/548M [00:07<00:00, 74.3MB/s]
Loading weights: 100% 148/148 [00:00<00:00, 2115.20it/s, Materializing param=transformer.wte.weight]
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
generation_config.json: 100% 124/124 [00:00<00:00, 621kB/

## Smoke Test 2: Testing using HC3 Unified dataset
This test ensures that the pipeline works with a different dataset (`hc3_all`). It evaluates another 50 samples using similar settings to the first smoke test.

In [7]:
!python run.py --dataset hc3_all --base_model_name gpt2 --mask_filling_model_name t5-small --n_samples 50 --n_perturbation_list 3 --skip_baselines --batch_size 10 --chunk_size 10 --cache_dir hf_cache --output_name smoke_hc3

Using device: cuda
Saving results to absolute path: /content/drive/MyDrive/CSCI 544: Applied Natural Language Processing/Project/DetectGPT/tmp_results/smoke_hc3/gpt2-t5-small-temp/2026-04-13-06-58-35-361222-fp32-0.3-1-hc3_all-50
Using cache dir hf_cache
Loading BASE model gpt2...
Loading weights: 100% 148/148 [00:08<00:00, 18.37it/s, Materializing param=transformer.wte.weight]
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading mask filling model t5-small...
Loading weights: 100% 131/131 [00:01<00:00, 95.72it/s, Materializing param=shared.weight]
MOVING BASE MODEL TO GPU...DONE (3.76s)
Loading dataset hc3_all...
Token indices sequence length is longer than the specified maximum sequence length for this model (553 > 512). Running this sequence through the

## Medium Test (using HC3 Unified)
Once you confirm the smoke tests work, you can run a larger experiment.

**Note:** The command in the next cell is commented out (starts with `#`). To run it, remove the `#` symbol.

In [ ]:
!python run.py --output_name main --dataset hc3_all --base_model_name gpt2-medium --mask_filling_model_name t5-base --n_perturbation_list 10 --n_samples 500 --pct_words_masked 0.3 --span_length 2 --cache_dir hf_cache

# Calculating Metrics
After generating the perturbations and scoring them in the steps above, we need to calculate the final evaluation metrics (like ROC AUC).

To calculate the metrics, you will need the outputted json file from running DetectGPT. They will usually look like this:

```perturbation_3_z_results.json```

⚠️ **IMPORTANT FOR NEW USERS:** The `--path` argument in the cell below currently points to a specific result file. **You must update this path** to match the actual `.json` file generated in your `results` folder from the runs you just completed above. This command assumes that the results folder is in the project working directory.

In [10]:
# This is for Smoke Test 2. I just copied the output to find the directory and inputted here.
!python calculate_metrics.py --path results/smoke_hc3/gpt2-t5-small-temp/2026-04-13-06-58-35-361222-fp32-0.3-1-hc3_all-50/perturbation_3_z_results.json

Selected threshold for target_fpr(0.10): 6.0807 (FPR=0.1000, TPR=0.2200)
ROC-AUC: 0.6188
TPR@1%FPR: 0.02
threshold: 6.080661760496236
confusion matrix [[TN, FP], [FN, TP]] = [[np.int64(45), np.int64(5)], [np.int64(39), np.int64(11)]]
accuracy: 0.56 precision: 0.6875 recall: 0.22 f1: 0.3333333333333333
